# Wildlife Re-ID — Submission #2: k-reciprocal re-ranking + viewpoint compatibility + SIFT/RANSAC

Run top to bottom. Sections: **0** config · **1** data, folds & Sub #1 config · **2** k-reciprocal re-ranking · **3** viewpoint compatibility ·
**4** cached parallel top-M SIFT + RANSAC · **5** blending, CV tuning & ablation · **6** final fit, sanity checks & `submission_v2.csv` (+ diagnostics).

Needs from Sub #1 (same working directory): the folds CSV and `best_config.json`.
Rules reminder: only provided features, species never mixed, classical matching only (numpy/scipy/sklearn/opencv).

## 0. Config

In [1]:
import os, json, time, glob, warnings
from collections import Counter
from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mp
import numpy as np
import pandas as pd
import cv2
from scipy import sparse
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore", category=DeprecationWarning)

# ---------- directory layout ----------
# reid_student_data_main/ : train.csv, test.csv, miewid_features.npy, miewid_ids.csv, sample_submission.csv
# sift_features_train.npz, sift_features_test.npz : next to that folder (file, or an unzipped folder of <id>_xy.npy / <id>_desc.npy)
def _resolve_data_dir():
    cands = [os.environ.get("DATA_DIR"),r"C:\Users\Mayank\Downloads\projectwild3\reid_student_data_main",
             r"C:\Users\Mayank\Downloads\projectwild3", "."]
    for c in cands:
        if c and os.path.isfile(os.path.join(c, "train.csv")):
            return c
    raise FileNotFoundError("train.csv not found - set the DATA_DIR environment variable to the folder that contains it.")

DATA_DIR = _resolve_data_dir()
PARENT_DIR = os.path.dirname(os.path.abspath(DATA_DIR))

def find_file(name):
    for d in [".", DATA_DIR, PARENT_DIR]:
        p = os.path.join(d, name)
        if os.path.isfile(p):
            return p
    return None

# ---------- Sub #1 config ----------
NEW        = "new_individual"
SPECIES    = ["hyena", "leopard"]
N_FOLDS    = 5
SEED       = 42
BURST_COS  = None                      # must be the value used when folds were created in Sub #1
FOLDS_PATH = find_file(f"folds_burst{BURST_COS}.csv")
if FOLDS_PATH is None:                 # fall back to any folds file written by Sub #1
    _c = sorted({p for d in [".", DATA_DIR, PARENT_DIR] for p in glob.glob(os.path.join(d, "folds_burst*.csv"))}, key=os.path.getmtime)
    if not _c:
        raise FileNotFoundError("No folds_burst*.csv found - run Sub #1 first (it writes the folds file).")
    FOLDS_PATH = _c[-1]
    print("WARNING: folds_burst%s.csv not found, using newest folds file:" % BURST_COS, FOLDS_PATH)
BEST_CFG_PATH = find_file("best_config.json")
if BEST_CFG_PATH is None:
    raise FileNotFoundError("best_config.json not found - run Sub #1 first.")
CACHE_DIR = "cache"
OUT_PATH  = "submission.csv"

# ---------- re-ranking / SIFT / blending knobs ----------
RERANK_K1, RERANK_K2, RERANK_LAM = 20, 6, 0.3    # k-reciprocal re-ranking (Zhong et al.)
TOP_M_SIFT    = 100          # same-species gallery images per query that get SIFT+RANSAC
SIFT_RATIO    = 0.78
RANSAC_THRESH = 5.0
RANSAC_ITERS  = 2000
N_JOBS        = -1
CAND_K        = 200          # candidate gallery images per query kept for blending (top by S_rerank; must be >= TOP_M_SIFT)
assert CAND_K >= TOP_M_SIFT

print("DATA_DIR      :", DATA_DIR)
print("FOLDS_PATH    :", FOLDS_PATH)
print("BEST_CFG_PATH :", BEST_CFG_PATH)

DATA_DIR      : C:\Users\Mayank\Downloads\projectwild3\reid_student_data_main
FOLDS_PATH    : .\folds_burst0.8.csv
BEST_CFG_PATH : .\best_config.json


## 1. Load data, folds & Sub #1 config

In [2]:
def load_data(d=DATA_DIR):
    """Returns train df, test df, MiewID rows aligned to train, MiewID rows aligned to test."""
    tr = pd.read_csv(f"{d}/train.csv")
    te = pd.read_csv(f"{d}/test.csv")
    ids = pd.read_csv(f"{d}/miewid_ids.csv")
    col = "id" if "id" in ids.columns else ids.columns[0]
    F = np.load(f"{d}/miewid_features.npy").astype(np.float32)
    pos = pd.Series(np.arange(len(ids)), index=ids[col].values)
    Xtr, Xte = F[pos.loc[tr.id].values], F[pos.loc[te.id].values]   # KeyError if ids misalign
    tr["viewpoint"] = tr["viewpoint"].fillna("unk")
    te["viewpoint"] = te["viewpoint"].fillna("unk")
    return tr, te, Xtr, Xte

tr, te, Xtr, Xte = load_data()
print(tr.shape, te.shape, Xtr.shape, Xte.shape)

folds = pd.read_csv(FOLDS_PATH)
assert (folds["id"].values == tr["id"].values).all(), "folds file does not match train.csv - rebuild it with Sub #1"
_bc = json.load(open(BEST_CFG_PATH))
cfg, TAUS_V1 = _bc["cfg"], _bc["tau"]
print("Sub #1 cfg:", cfg, "| Sub #1 CV MAP@5:", round(_bc["cv_map"], 4), "| taus:", {k: round(v, 3) for k, v in TAUS_V1.items()})
for k in range(N_FOLDS):
    r = folds[f"role_f{k}"]
    print(f"fold {k}: train={(r=='train').sum()} val_known={(r=='val_known').sum()} val_new={(r=='val_new').sum()}")

(7163, 4) (2791, 3) (7163, 2152) (2791, 2152)
Sub #1 cfg: {'agg': 'max', 'center': True, 'pca': 512, 'dba': 0, 'qe': 0, 'vp': 0.0} | Sub #1 CV MAP@5: 0.8366 | taus: {'hyena': 0.173, 'leopard': 0.254}
fold 0: train=4983 val_known=1415 val_new=765
fold 1: train=5335 val_known=1215 val_new=613
fold 2: train=5083 val_known=1176 val_new=904
fold 3: train=5444 val_known=1183 val_new=536
fold 4: train=4968 val_known=1381 val_new=814


In [3]:
# ---------- helpers copied from Sub #1 (unchanged) ----------
def map_at_5(y_true, y_pred_lists):
    tot = 0.0
    for t, p in zip(y_true, y_pred_lists):
        p = list(dict.fromkeys(p))[:5]
        if t in p:
            tot += 1.0 / (p.index(t) + 1)
    return tot / len(y_true)


def l2n(X):
    return X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-12)


def species_embeddings(Xtr_sp, Xte_sp, cfg):
    X = l2n(np.vstack([Xtr_sp, Xte_sp]).astype(np.float32))
    if cfg["center"]:
        X = l2n(X - X.mean(0, keepdims=True))
    if cfg["pca"]:
        X = l2n(PCA(cfg["pca"], whiten=True, random_state=0).fit_transform(X).astype(np.float32))
    return X[:len(Xtr_sp)], X[len(Xtr_sp):]


def expand(Q, G, k, alpha=3.0, include_q=True):
    S = Q @ G.T
    idx = np.argpartition(-S, k - 1, axis=1)[:, :k]
    w = np.clip(np.take_along_axis(S, idx, 1), 0, None) ** alpha
    out = (w[:, :, None] * G[idx]).sum(1)
    if include_q:
        out = out + Q
    return l2n(out)


def identity_scores(S, gcode, agg):
    order = np.argsort(gcode, kind="stable")
    Ss, c = S[:, order], gcode[order]
    starts = np.r_[0, np.flatnonzero(np.diff(c)) + 1]
    ends = np.r_[starts[1:], len(c)]
    if agg == "max":
        return np.maximum.reduceat(Ss, starts, axis=1)
    k = int(agg[3:])
    out = np.empty((S.shape[0], len(starts)), np.float32)
    for j, (a, b) in enumerate(zip(starts, ends)):
        kk = min(k, b - a)
        out[:, j] = np.partition(Ss[:, a:b], -kk, axis=1)[:, -kk:].mean(1)
    return out


def rank_topk(Q, G, gcode, vq, vg, cfg, topn=5):
    if cfg["dba"] > 0:
        G = expand(G, G, cfg["dba"], include_q=False)
    if cfg["qe"] > 0:
        Q = expand(Q, G, cfg["qe"], include_q=True)
    S = Q @ G.T
    if cfg["vp"] > 0:
        S = S + cfg["vp"] * (vq[:, None] == vg[None, :])
    sc = identity_scores(S, gcode, cfg["agg"])
    idx = np.argsort(-sc, axis=1)[:, :min(topn, sc.shape[1])]
    return idx, np.take_along_axis(sc, idx, 1)


def new_position(top_sc, tau):
    return np.minimum((top_sc >= tau).sum(1), 4)


def apply_new(top_names, top_sc, tau):
    pos = new_position(top_sc, tau)
    out = []
    for names, p in zip(top_names, pos):
        lst = list(names)
        lst.insert(int(p), NEW)
        out.append(lst[:5])
    return out


def score_samples(top_sc, has, j, isnew, tau):
    pos = new_position(top_sc, tau)
    r = np.where(isnew, pos, j + (pos <= j))
    ok = (isnew | has) & (r < 5)
    return np.where(ok, 1.0 / (r + 1), 0.0)


def tune_tau(top_sc, has, j, isnew):
    q = np.quantile(top_sc.ravel(), np.linspace(0, 1, 401))
    grid = np.r_[-np.inf, np.unique(q), np.inf]
    maps = np.array([score_samples(top_sc, has, j, isnew, t).mean() for t in grid])
    b = int(maps.argmax())
    return float(grid[b]), float(maps[b]), float(maps[0])


# ---------- Sub #1 pipeline (needed only to reproduce the baseline row of the ablation table) ----------
vp_map = {v: i for i, v in enumerate(sorted(set(tr.viewpoint) | set(te.viewpoint)))}
def vp_codes(df):
    return np.where(df.viewpoint.values == "unk", -1, df.viewpoint.map(vp_map).values)
vp_tr, vp_te = vp_codes(tr), vp_codes(te)
gal_vp = lambda v: np.where(v < 0, -2, v)

_cache = {}
def get_pre(sp, cfg):
    key = (sp, cfg["center"], cfg["pca"])
    if key not in _cache:
        m_tr, m_te = (tr.species == sp).values, (te.species == sp).values
        _cache[key] = species_embeddings(Xtr[m_tr], Xte[m_te], cfg)[0]
    return _cache[key]

def collect_oof(cfg, folds=folds):
    res = {}
    for sp in SPECIES:
        P = get_pre(sp, cfg)
        sub = np.flatnonzero((tr.species == sp).values)
        loc = np.full(len(tr), -1); loc[sub] = np.arange(len(sub))
        T, H, J, N, FD = [], [], [], [], []
        for k in range(N_FOLDS):
            role = folds[f"role_f{k}"].values
            gi, vi = sub[role[sub] == "train"], sub[role[sub] != "train"]
            classes, gcode = np.unique(tr.identity.values[gi], return_inverse=True)
            top, sc = rank_topk(P[loc[vi]], P[loc[gi]], gcode, vp_tr[vi], gal_vp(vp_tr[gi]), cfg)
            isnew = role[vi] == "val_new"
            cm = {c: i for i, c in enumerate(classes)}
            y = np.array([-1 if n else cm[i] for i, n in zip(tr.identity.values[vi], isnew)])
            match = top == y[:, None]
            T.append(sc); H.append(match.any(1)); J.append(match.argmax(1)); N.append(isnew); FD.append(np.full(len(vi), k))
        res[sp] = tuple(map(np.concatenate, (T, H, J, N, FD)))
    return res

def run_cfg(cfg):
    out, allsc = {"species": {}}, []
    for sp, (T, H, J, N, FD) in collect_oof(cfg).items():
        tau, m, m_no = tune_tau(T, H, J, N)
        s = score_samples(T, H, J, N, tau)
        out["species"][sp] = dict(tau=tau, map=m, n=len(T))
        allsc.append(s)
    out["map"] = float(np.concatenate(allsc).mean())
    return out

## 2. Transductive k-reciprocal re-ranking (Zhong et al., 2017)
Gallery and queries are stacked (CV: train-fold + val-fold; inference: train + test), so query-to-query affinities are used
(the same `new_individual` can appear several times in test). Jaccard distances are computed sparsely; full N×N arrays are only
held as float32 and processed in chunks.

In [4]:
def k_reciprocal_rerank(Q, G, k1=RERANK_K1, k2=RERANK_K2, lam=RERANK_LAM, chunk=1024):
    """Q (nq,d), G (ng,d): L2-normalised rows.  Returns (S_rerank (nq,ng), S_cos (nq,ng), max_q2q_sim (nq,)).
    S_rerank = 1 - [(1-lam)*jaccard_dist + lam*original_dist];  max_q2q_sim = max cosine to OTHER queries."""
    Q = np.ascontiguousarray(Q, dtype=np.float32); G = np.ascontiguousarray(G, dtype=np.float32)
    nq, ng = len(Q), len(G)
    X = np.vstack([Q, G]); N = nq + ng
    S_cos = Q @ G.T
    Sqq = Q @ Q.T
    np.fill_diagonal(Sqq, -np.inf)
    max_q2q = Sqq.max(1).astype(np.float32) if nq > 1 else np.full(nq, -1.0, np.float32)
    del Sqq

    kk = min(k1, N - 1)                                   # neighbour lists have kk+1 entries (self first)
    D = np.empty((N, N), np.float32)                      # row-max-normalised squared euclidean distance
    nn = np.empty((N, kk + 1), np.int32)
    for a in range(0, N, chunk):
        b = min(a + chunk, N)
        d = 2.0 - 2.0 * (X[a:b] @ X.T)
        np.maximum(d, 0, out=d)
        d /= np.maximum(d.max(1, keepdims=True), 1e-12)
        D[a:b] = d
        part = np.argpartition(d, kk, axis=1)[:, :kk + 1]
        o = np.argsort(np.take_along_axis(d, part, 1), axis=1)
        nn[a:b] = np.take_along_axis(part, o, 1)
    ar = np.arange(N)
    for i in np.flatnonzero(nn[:, 0] != ar):             # exact duplicates can displace self from position 0
        w = np.flatnonzero(nn[i] == i)
        if len(w):
            nn[i, [0, w[0]]] = nn[i, [w[0], 0]]
        else:
            nn[i] = np.r_[i, nn[i, :-1]]

    # k-reciprocal neighbour sets (vectorised) ...
    rec = (nn[nn] == ar[:, None, None]).any(2)
    h = min(int(round(k1 / 2)), kk)
    nnh = nn[:, :h + 1]
    rech = (nnh[nnh] == ar[:, None, None]).any(2)
    Rfull = [nn[i][rec[i]] for i in range(N)]
    Rhalf = [set(nnh[i][rech[i]].tolist()) for i in range(N)]
    # ... expanded with the neighbours' half-sets that overlap >2/3, then Gaussian-weighted into sparse V
    rows, cols, vals = [], [], []
    for i in range(N):
        R = Rfull[i]
        Rset = set(R.tolist())
        add = set()
        for c in R.tolist():
            rc = Rhalf[c]
            if rc and sum(1 for x in rc if x in Rset) > (2.0 / 3.0) * len(rc):
                add |= rc
        idx = np.fromiter(Rset | add, dtype=np.int64)
        w = np.exp(-D[i, idx])
        rows.append(np.full(len(idx), i, np.int64)); cols.append(idx); vals.append((w / w.sum()).astype(np.float32))
    V = sparse.csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))), shape=(N, N))
    if k2 > 1:                                            # local query expansion: mean of V over the k2 nearest rows
        k2e = min(k2, kk + 1)
        A = sparse.csr_matrix((np.full(N * k2e, 1.0 / k2e, np.float32), (np.repeat(ar, k2e), nn[:, :k2e].ravel())), shape=(N, N))
        V = (A @ V).tocsr()
    Vc = V.tocsc()
    rp, ri, rd = V.indptr, V.indices, V.data
    cp, ci, cd = Vc.indptr, Vc.indices, Vc.data

    J = np.empty((nq, ng), np.float32)                    # jaccard distance, queries x gallery
    for i in range(nq):
        cs, vs = ri[rp[i]:rp[i + 1]], rd[rp[i]:rp[i + 1]]
        st, en = cp[cs].astype(np.int64), cp[cs + 1].astype(np.int64)
        cnt = en - st
        tot = int(cnt.sum())
        gather = np.arange(tot) + np.repeat(st - (np.cumsum(cnt) - cnt), cnt)
        tm = np.bincount(ci[gather], weights=np.minimum(cd[gather], np.repeat(vs, cnt)), minlength=N)   # sum_k min(V_ik, V_jk)
        J[i] = (1.0 - tm / (2.0 - tm))[nq:]
    S_rerank = 1.0 - ((1.0 - lam) * J + lam * D[:nq, nq:])
    return S_rerank.astype(np.float32), S_cos, max_q2q

## 3. Viewpoint compatibility (4 buckets)
`same` · `adjacent` (left↔frontleft/backleft, right↔frontright/backright, front↔frontleft/frontright, back↔backleft/backright) ·
`opposite` (left↔right, front↔back, up↔down) · `missing` (either side `unk`). Any other pair gets bucket `other` (bonus fixed at 0).

In [5]:
BK_SAME, BK_ADJ, BK_OPP, BK_MISSING, BK_OTHER = 0, 1, 2, 3, 4

_ADJ = set()
for a, bs in {"left": ["frontleft", "backleft"], "right": ["frontright", "backright"],
              "front": ["frontleft", "frontright"], "back": ["backleft", "backright"]}.items():
    for b in bs:
        _ADJ |= {(a, b), (b, a)}
_OPP = set()
for a, b in [("left", "right"), ("front", "back"), ("up", "down")]:
    _OPP |= {(a, b), (b, a)}

def vp_bucket(vq, vg):
    if vq == "unk" or vg == "unk":
        return BK_MISSING
    if vq == vg:
        return BK_SAME
    if (vq, vg) in _ADJ:
        return BK_ADJ
    if (vq, vg) in _OPP:
        return BK_OPP
    return BK_OTHER

VP_VOCAB = sorted(set(tr.viewpoint) | set(te.viewpoint) | {"unk"})
VP_IDX = {v: i for i, v in enumerate(VP_VOCAB)}
VP_TABLE = np.array([[vp_bucket(a, b) for b in VP_VOCAB] for a in VP_VOCAB], dtype=np.int8)   # (K, K) lookup
vp_id_tr = tr.viewpoint.map(VP_IDX).values.astype(np.int16)
vp_id_te = te.viewpoint.map(VP_IDX).values.astype(np.int16)
print("viewpoints:", VP_VOCAB)
print("bucket share among train x train pairs:",
      dict(zip(["same", "adjacent", "opposite", "missing", "other"],
               np.round(np.bincount(VP_TABLE[vp_id_tr[:, None], vp_id_tr[None, :2000]].ravel(), minlength=5) / (len(tr) * 2000), 3))))

viewpoints: ['back', 'backleft', 'backright', 'down', 'front', 'frontleft', 'frontright', 'left', 'right', 'unk', 'up']
bucket share among train x train pairs: {'same': 0.263, 'adjacent': 0.031, 'opposite': 0.262, 'missing': 0.036, 'other': 0.408}


## 4. Cached, parallel top-M SIFT + RANSAC verification
1. Re-rank every CV fold (val vs. train-fold) and the test set (test vs. full train), per species; keep the top `CAND_K` candidates per query
   (sorted by `S_rerank`) together with `S_rerank`, `S_cos`, viewpoint bucket and identity code.
2. The first `TOP_M_SIFT` candidates of every query get ratio-test + mutual-NN matching and RANSAC (partial-affine) verification.
3. Results are cached in `cache/sift_oof_top100.parquet` / `cache/sift_test_top100.parquet` and reloaded when parameters and pair lists are unchanged.

In [6]:
# ---- per-species embeddings (Sub #1 preprocessing, no DBA/QE: re-ranking replaces them) ----
code_tr = np.full(len(tr), -1, np.int32)      # identity code (index into CLASSES[sp]) for every train row
LOC = np.full(len(tr), -1)                    # row inside the species-local embedding matrix
CLASSES, SUB, TE_SUB, EMB = {}, {}, {}, {}
for sp in SPECIES:
    m_tr, m_te = (tr.species == sp).values, (te.species == sp).values
    SUB[sp], TE_SUB[sp] = np.flatnonzero(m_tr), np.flatnonzero(m_te)
    EMB[sp] = species_embeddings(Xtr[m_tr], Xte[m_te], cfg)               # (P_train_sp, P_test_sp)
    CLASSES[sp], inv = np.unique(tr.identity.values[m_tr], return_inverse=True)
    code_tr[SUB[sp]] = inv
    LOC[SUB[sp]] = np.arange(m_tr.sum())


def make_block(Q, G, vq, vg, gcode_G, grow_G):
    Sr, Sc, q2q = k_reciprocal_rerank(Q, G)
    ng = G.shape[0]
    assert ng >= CAND_K, f"gallery has only {ng} images; lower CAND_K"
    part = np.argpartition(-Sr, CAND_K - 1, axis=1)[:, :CAND_K]
    cand = np.take_along_axis(part, np.argsort(-np.take_along_axis(Sr, part, 1), axis=1), 1)   # sorted by S_rerank desc
    return dict(Sr=np.take_along_axis(Sr, cand, 1), Sc=np.take_along_axis(Sc, cand, 1),
                bk=VP_TABLE[vq[:, None], vg[cand]], gc=gcode_G[cand].astype(np.int32),
                cand=grow_G[cand].astype(np.int32), q2q=q2q)


def cat(parts):
    return {k: np.concatenate([b[k] for b in parts]) for k in parts[0]}


def build_cv_pool(sp):
    P_tr, _ = EMB[sp]; sub = SUB[sp]
    parts = []
    for k in range(N_FOLDS):
        t0 = time.time()
        role = folds[f"role_f{k}"].values
        gi, vi = sub[role[sub] == "train"], sub[role[sub] != "train"]
        blk = make_block(P_tr[LOC[vi]], P_tr[LOC[gi]], vp_id_tr[vi], vp_id_tr[gi], code_tr[gi], gi)
        isnew = role[vi] == "val_new"
        blk.update(fold=np.full(len(vi), k), q_rows=vi, isnew=isnew, y=np.where(isnew, -1, code_tr[vi]).astype(np.int32))
        parts.append(blk)
        print(f"  {sp} fold {k}: queries={len(vi)} gallery={len(gi)}  [{time.time()-t0:.0f}s]")
    return cat(parts)


def build_test_pool(sp):
    P_tr, P_te = EMB[sp]; sub = SUB[sp]
    t0 = time.time()
    blk = make_block(P_te, P_tr, vp_id_te[TE_SUB[sp]], vp_id_tr[sub], code_tr[sub], sub)
    blk.update(fold=np.full(len(TE_SUB[sp]), -1), q_rows=TE_SUB[sp])
    print(f"  {sp} test: queries={len(TE_SUB[sp])} gallery={len(sub)}  [{time.time()-t0:.0f}s]")
    return blk


t0 = time.time()
print("re-ranking CV folds ...")
POOL_CV = {sp: build_cv_pool(sp) for sp in SPECIES}
print("re-ranking test ...")
POOL_TE = {sp: build_test_pool(sp) for sp in SPECIES}
print(f"done [{time.time()-t0:.0f}s]")

re-ranking CV folds ...
  hyena fold 0: queries=514 gallery=1398  [0s]
  hyena fold 1: queries=541 gallery=1371  [0s]
  hyena fold 2: queries=488 gallery=1424  [0s]
  hyena fold 3: queries=522 gallery=1390  [0s]
  hyena fold 4: queries=530 gallery=1382  [0s]
  leopard fold 0: queries=1666 gallery=3585  [1s]
  leopard fold 1: queries=1287 gallery=3964  [1s]
  leopard fold 2: queries=1592 gallery=3659  [1s]
  leopard fold 3: queries=1197 gallery=4054  [1s]
  leopard fold 4: queries=1665 gallery=3586  [1s]
re-ranking test ...
  hyena test: queries=1217 gallery=1912  [1s]
  leopard test: queries=1574 gallery=5251  [2s]
done [9s]


In [7]:
# ---------- SIFT source (file .npz, or an unzipped folder of <id>_xy.npy / <id>_desc.npy) ----------
def locate_sift(name):
    roots = [os.environ.get("SIFT_DIR"), PARENT_DIR, os.getcwd(), DATA_DIR]
    for r in roots:
        if not r:
            continue
        for nm in (name, name[:-4] if name.endswith(".npz") else name):
            p = os.path.join(r, nm)
            if os.path.exists(p):
                return p
    raise FileNotFoundError(f"{name} not found in {roots} - set SIFT_DIR to the folder that contains it.")


class SiftSource:
    def __init__(self, path):
        self.zs, self.dir, self.map = [], None, None
        if os.path.isfile(path):
            self.zs = [np.load(path)]
        else:
            npzs = sorted(glob.glob(os.path.join(path, "**", "*.npz"), recursive=True))
            if npzs:
                self.zs = [np.load(p) for p in npzs]
            else:
                self.dir = path
        print("SIFT source:", path, "| mode:", "npz" if self.zs else "folder of .npy")

    def get(self, key):
        for z in self.zs:
            try:
                return z[key]
            except KeyError:
                continue
        if self.dir is not None:
            p = os.path.join(self.dir, key + ".npy")
            if os.path.exists(p):
                return np.load(p)
            if self.map is None:
                self.map = {os.path.splitext(os.path.basename(q))[0]: q for q in glob.glob(os.path.join(self.dir, "**", "*.npy"), recursive=True)}
            return np.load(self.map[key])
        raise KeyError(key)


SIFT = None
def ensure_sift():
    global SIFT
    if SIFT is not None:
        return
    t0 = time.time()
    src_tr, src_te = SiftSource(locate_sift("sift_features_train.npz")), SiftSource(locate_sift("sift_features_test.npz"))
    SIFT = {}
    for src, ids in ((src_tr, tr.id.values), (src_te, te.id.values)):
        for n, i in enumerate(ids):
            SIFT[i] = (src.get(f"{i}_xy").reshape(-1, 2).astype(np.float32), src.get(f"{i}_desc").reshape(-1, 128))
            if (n + 1) % 3000 == 0:
                print(f"  loaded {n+1}/{len(ids)} SIFT arrays")
    print(f"SIFT loaded: {len(SIFT)} images [{time.time()-t0:.0f}s]")


# ---------- pair matching ----------
_BF = None
def _pair_stats(xq, dq32, xg, dg):
    """(ratio_matches, ransac_inliers) for one image pair."""
    if len(xq) < 4 or len(xg) < 4:
        return 0, 0
    dg32 = dg.astype(np.float32)
    knn = _BF.knnMatch(dq32, dg32, k=2)
    good = [m[0] for m in knn if len(m) == 2 and m[0].distance < SIFT_RATIO * m[1].distance]      # Lowe ratio test
    if not good:
        return 0, 0
    rev = _BF.match(dg32, dq32)                                                                    # mutual NN check
    rt = np.fromiter((r.trainIdx for r in rev), dtype=np.int64, count=len(rev))
    good = [m for m in good if rt[m.trainIdx] == m.queryIdx]
    if len(good) < 4:
        return len(good), 0
    pq = np.float32([xq[m.queryIdx] for m in good]); pg = np.float32([xg[m.trainIdx] for m in good])
    cv2.setRNGSeed(0)
    _, mask = cv2.estimateAffinePartial2D(pq, pg, method=cv2.RANSAC, ransacReprojThreshold=RANSAC_THRESH, maxIters=RANSAC_ITERS)
    return len(good), (0 if mask is None else int(mask.sum()))


def _sift_chunk(items):
    global _BF
    if _BF is None:
        cv2.setNumThreads(1)
        _BF = cv2.BFMatcher(cv2.NORM_L2)
    out = []
    for qid, gids in items:
        xq, dq = SIFT[qid]
        dq32 = dq.astype(np.float32)
        for gid in gids:
            xg, dg = SIFT[gid]
            nm, ni = _pair_stats(xq, dq32, xg, dg)
            out.append((len(xq), len(xg), nm, ni))
    return np.array(out, dtype=np.int32).reshape(-1, 4)


def run_sift(q_ids, g_mat, label, chunk_q=40):
    """q_ids (nq,), g_mat (nq, M) gallery ids -> int32 array (nq*M, 4): n_kp_q, n_kp_g, ratio_matches, ransac_inliers."""
    ensure_sift()
    tasks = [[(q_ids[i], g_mat[i].tolist()) for i in range(a, min(a + chunk_q, len(q_ids)))] for a in range(0, len(q_ids), chunk_q)]
    workers = os.cpu_count() if N_JOBS in (-1, None) else int(N_JOBS)
    workers = max(1, workers)
    t0, res = time.time(), []
    try:
        ctx = mp.get_context("fork")
    except ValueError:
        ctx, workers = None, 1
    print(f"[{label}] {len(q_ids)} queries x {g_mat.shape[1]} gallery = {len(q_ids)*g_mat.shape[1]:,} pairs on {workers} worker(s)")
    if workers == 1:
        for n, t in enumerate(tasks):
            res.append(_sift_chunk(t))
            if (n + 1) % max(1, len(tasks) // 20) == 0:
                print(f"  {n+1}/{len(tasks)} chunks [{time.time()-t0:.0f}s]")
    else:
        with ProcessPoolExecutor(max_workers=workers, mp_context=ctx) as ex:
            for n, r in enumerate(ex.map(_sift_chunk, tasks)):
                res.append(r)
                if (n + 1) % max(1, len(tasks) // 20) == 0:
                    print(f"  {n+1}/{len(tasks)} chunks [{time.time()-t0:.0f}s]")
    print(f"[{label}] done [{time.time()-t0:.0f}s]")
    return np.concatenate(res)

In [8]:
M_SIFT = min(TOP_M_SIFT, CAND_K)
os.makedirs(CACHE_DIR, exist_ok=True)
META_PATH = os.path.join(CACHE_DIR, "sift_meta.json")
SIG = dict(M=M_SIFT, ratio=SIFT_RATIO, ransac=RANSAC_THRESH, iters=RANSAC_ITERS, k1=RERANK_K1, k2=RERANK_K2, lam=RERANK_LAM,
           center=cfg["center"], pca=cfg["pca"], cand_k=CAND_K, n_train=len(tr), n_test=len(te))


def make_pairs(pool, sp, q_id_arr):
    g = pool["cand"][:, :M_SIFT]
    return pd.DataFrame(dict(fold=np.repeat(pool["fold"], M_SIFT), query_id=np.repeat(q_id_arr[pool["q_rows"]], M_SIFT),
                             gallery_id=tr.id.values[g].ravel(), gallery_identity=tr.identity.values[g].ravel(), species=sp))


def _save_tbl(df, path):
    try:
        df.to_parquet(path, index=False)
    except Exception as e:                      # no parquet engine -> compressed csv
        path = path.replace(".parquet", ".csv.gz"); df.to_csv(path, index=False)
        print("  (parquet unavailable: %s) saved %s" % (type(e).__name__, path))


def _load_tbl(path):
    if os.path.exists(path):
        return pd.read_parquet(path)
    if os.path.exists(path.replace(".parquet", ".csv.gz")):
        return pd.read_csv(path.replace(".parquet", ".csv.gz"))
    return None


def get_sift_table(kind, pairs):
    path = os.path.join(CACHE_DIR, f"sift_{kind}_top{TOP_M_SIFT}.parquet")
    meta = json.load(open(META_PATH)) if os.path.exists(META_PATH) else {}
    cached = _load_tbl(path)
    if (cached is not None and meta.get(kind) == SIG and len(cached) == len(pairs)
            and (cached.query_id.values == pairs.query_id.values).all() and (cached.gallery_id.values == pairs.gallery_id.values).all()):
        print(f"[{kind}] loaded cache {path} ({len(cached):,} pairs)")
        return cached
    st = run_sift(pairs.query_id.values[::M_SIFT], pairs.gallery_id.values.reshape(-1, M_SIFT), kind)
    out = pairs.copy()
    out["n_kp_q"], out["n_kp_g"], out["ratio_matches"], out["ransac_inliers"] = st.T
    out["inlier_ratio"] = out.ransac_inliers / np.maximum(out.ratio_matches, 1)
    out = out[["fold", "query_id", "gallery_id", "gallery_identity", "species", "n_kp_q", "n_kp_g", "ratio_matches", "ransac_inliers", "inlier_ratio"]]
    _save_tbl(out, path)
    meta[kind] = SIG
    json.dump(meta, open(META_PATH, "w"), indent=1)
    return out


def attach_inliers(pools, tbl):
    off = 0
    for sp in SPECIES:
        P = pools[sp]; n = len(P["cand"])
        P["inl"] = np.zeros((n, CAND_K), np.float32)
        P["inl"][:, :M_SIFT] = tbl.ransac_inliers.values[off:off + n * M_SIFT].reshape(n, M_SIFT)
        off += n * M_SIFT
    assert off == len(tbl)


pairs_oof = pd.concat([make_pairs(POOL_CV[sp], sp, tr.id.values) for sp in SPECIES], ignore_index=True)
pairs_te  = pd.concat([make_pairs(POOL_TE[sp], sp, te.id.values) for sp in SPECIES], ignore_index=True)
sift_oof  = get_sift_table("oof", pairs_oof)
sift_te   = get_sift_table("test", pairs_te)
attach_inliers(POOL_CV, sift_oof)
attach_inliers(POOL_TE, sift_te)

# quick look: do true same-identity pairs get more inliers than other pairs? (OOF known queries only)
_y = np.concatenate([np.repeat(POOL_CV[sp]["y"], M_SIFT) for sp in SPECIES])
_g = np.concatenate([POOL_CV[sp]["gc"][:, :M_SIFT].ravel() for sp in SPECIES])
_known = np.concatenate([np.repeat(~POOL_CV[sp]["isnew"], M_SIFT) for sp in SPECIES])
_same = (_y == _g) & _known
_inl = sift_oof.ransac_inliers.values
print(f"median inliers  same-identity pairs: {np.median(_inl[_same]):.0f} | other pairs (known queries): {np.median(_inl[_known & ~_same]):.0f} | "
      f"pairs with 0 inliers: {(_inl == 0).mean():.1%}")

[oof] loaded cache cache\sift_oof_top100.parquet (1,000,200 pairs)
SIFT source: C:\Users\Mayank\Downloads\projectwild3\sift_features_train.npz | mode: folder of .npy
SIFT source: C:\Users\Mayank\Downloads\projectwild3\sift_features_test.npz | mode: folder of .npy
  loaded 3000/7163 SIFT arrays
  loaded 6000/7163 SIFT arrays
SIFT loaded: 9954 images [119s]
[test] 2791 queries x 100 gallery = 279,100 pairs on 1 worker(s)
  3/70 chunks [61s]
  6/70 chunks [126s]
  9/70 chunks [190s]
  12/70 chunks [251s]
  15/70 chunks [315s]
  18/70 chunks [379s]
  21/70 chunks [450s]
  24/70 chunks [518s]
  27/70 chunks [580s]
  30/70 chunks [646s]
  33/70 chunks [730s]
  36/70 chunks [816s]
  39/70 chunks [899s]
  42/70 chunks [975s]
  45/70 chunks [1048s]
  48/70 chunks [1123s]
  51/70 chunks [1202s]
  54/70 chunks [1277s]
  57/70 chunks [1359s]
  60/70 chunks [1442s]
  63/70 chunks [1522s]
  66/70 chunks [1606s]
  69/70 chunks [1687s]
[test] done [1708s]
median inliers  same-identity pairs: 2 | other

## 5. Blending, CV tuning & ablation table
$$\text{score}(q,g) = w_{\text{rerank}} S_{\text{rerank}} + w_{\text{cos}} S_{\text{cos}} + w_{\text{sift}} \log(1+\text{inliers}) + b_{\text{vp}}$$
Image-pair scores are aggregated to identities (`max`/`topN`), then `new_individual` is inserted at a per-species threshold τ (tuned on pooled OOF MAP@5).
Scale is fixed by `w_cos = 1 - w_rerank` (the overall scale is absorbed by τ). Tuning = coordinate search with shrinking steps (the objective is a step function,
so gradient-free coordinate moves are more reliable than a simplex). In the final stage two extra discrete choices are tried: the aggregation, and an inlier
floor `f` in `log(1 + max(inliers - f, 0))` (`f = 0` is the formula above; RANSAC on ≥4 matches always yields ≥2 inliers, so a floor can remove that noise level).

In [9]:
NEG = -1e6

def agg_topk(sc, gc, agg="max", topn=5):
    """sc, gc (nq, K): candidate-image scores / identity codes -> top identities (codes, scores), (nq, topn), best first.
    Identity score = mean of its best k image scores among the candidates (k=1 for 'max', 2 for 'top2', ...)."""
    nq, K = sc.shape
    k = 1 if agg == "max" else int(agg[3:])
    o1 = np.argsort(-sc, axis=1)
    gc1 = np.take_along_axis(gc, o1, 1)
    o2 = np.argsort(gc1, axis=1, kind="stable")            # group by identity, best image first inside a group
    order = np.take_along_axis(o1, o2, 1)
    A, V = np.take_along_axis(gc, order, 1), np.take_along_axis(sc, order, 1)
    acc, cnt = V.astype(np.float32).copy(), np.ones((nq, K), np.float32)
    for j in range(1, k):
        same = A[:, j:] == A[:, :K - j]
        acc[:, :K - j] += np.where(same, V[:, j:], 0)
        cnt[:, :K - j] += same
    first = np.ones((nq, K), bool); first[:, 1:] = A[:, 1:] != A[:, :-1]
    gs = np.where(first, acc / cnt, NEG).astype(np.float32)
    kk = min(topn, K)
    part = np.argpartition(-gs, kk - 1, axis=1)[:, :kk]
    sel = np.take_along_axis(part, np.argsort(-np.take_along_axis(gs, part, 1), axis=1), 1)
    ids, scs = np.take_along_axis(A, sel, 1), np.take_along_axis(gs, sel, 1)
    return np.where(scs > NEG / 2, ids, -2), scs               # -2 = padding (fewer than topn identities among candidates)


def blend(P, p):
    btab = np.array([p["b_same"], p["b_adj"], p["b_opp"], 0.0, 0.0], np.float32)       # same, adjacent, opposite, missing, other
    lsift = np.log1p(np.maximum(P["inl"] - p["floor"], 0))
    return p["w_rerank"] * P["Sr"] + (1 - p["w_rerank"]) * P["Sc"] + p["w_sift"] * lsift + btab[P["bk"]]


def eval_blend(p, pools=None, detail=False):
    pools = POOL_CV if pools is None else pools
    res, allsc = {}, []
    for sp in SPECIES:
        P = pools[sp]
        top, tsc = agg_topk(blend(P, p), P["gc"], p["agg"])
        match = top == P["y"][:, None]
        H, J, N = match.any(1), match.argmax(1), P["isnew"]
        tau, m, m_no = tune_tau(tsc, H, J, N)
        s = score_samples(tsc, H, J, N, tau)
        res[sp] = dict(tau=tau, map=float(m), map_new_last=float(m_no), recall5=float(H[~N].mean()), new1=float((new_position(tsc, tau) == 0).mean()))
        if detail:
            res[sp].update(tsc=tsc, H=H, J=J, N=N)
        allsc.append(s)
    res["map"] = float(np.concatenate(allsc).mean())
    return res


BOUNDS = dict(w_rerank=(0.0, 1.0), w_sift=(0.0, 2.0), b_same=(-1.0, 1.0), b_adj=(-1.0, 1.0), b_opp=(-1.0, 1.0))
STEP0  = dict(w_rerank=0.25, w_sift=0.05, b_same=0.05, b_adj=0.05, b_opp=0.05)

def coord_search(p0, free, tag="", min_div=16, max_rounds=40):
    p, best = dict(p0), eval_blend(p0)["map"]
    steps, n_ev = {k: STEP0[k] for k in free}, 1
    for rnd in range(max_rounds):
        improved = False
        for k in free:
            for d in (+1, -1):
                while True:
                    q = dict(p); q[k] = float(np.clip(p[k] + d * steps[k], *BOUNDS[k]))
                    if q[k] == p[k]:
                        break
                    m = eval_blend(q)["map"]; n_ev += 1
                    if m > best + 1e-5:
                        p, best, improved = q, m, True
                    else:
                        break
        if not improved:
            steps = {k: v / 2 for k, v in steps.items()}
            if all(steps[k] < STEP0[k] / min_div for k in free):
                break
    print(f"  [{tag}] {n_ev} evals -> MAP {best:.4f}  " + "  ".join(f"{k}={p[k]:.3f}" for k in free) + f"  agg={p['agg']} floor={p['floor']}")
    return p, best

In [10]:
t0 = time.time()
ablation = []
def add_row(name, res):
    ablation.append(dict(stage=name, **{sp: res[sp]["map"] for sp in SPECIES}, overall=res["map"]))

# ---- 1. Sub #1 MiewID baseline (exact Sub #1 pipeline, full gallery) ----
r1 = run_cfg(cfg)
add_row("1. Sub #1 MiewID baseline", {**{sp: r1["species"][sp] for sp in SPECIES}, "map": r1["map"]})
print(f"Sub #1 baseline reproduced: {r1['map']:.4f}  (best_config.json says {_bc['cv_map']:.4f})")

p0 = dict(w_rerank=0.5, w_sift=0.0, b_same=float(cfg["vp"]) if cfg["vp"] else 0.05, b_adj=0.0, b_opp=0.0, agg=cfg["agg"], floor=0)

# ---- 2. + k-reciprocal re-ranking (blend rerank/cos, Sub #1 exact-match viewpoint bonus only) ----
p2, _ = coord_search(p0, ["w_rerank", "b_same"], "2: rerank + exact vp")
r2 = eval_blend(p2); add_row("2. + k-Reciprocal Re-Ranking", r2)

# ---- 3. + 4-bucket viewpoint compatibility ----
p3_0 = {**p2, "b_adj": p2["b_same"] / 2}
p3, _ = coord_search(p3_0, ["w_rerank", "b_same", "b_adj", "b_opp"], "3: + vp buckets")
r3 = eval_blend(p3); add_row("3. + 4-Bucket Viewpoint Compatibility", r3)

# ---- 4. + SIFT + RANSAC (also tries inlier floor and aggregation) ----
free4 = ["w_rerank", "w_sift", "b_same", "b_adj", "b_opp"]
best_p, best_m = None, -1
for fl in (0, 3, 5, 8):
    cand, m = coord_search({**p3, "w_sift": 0.05, "floor": fl}, free4, f"4: floor={fl}")
    if m > best_m:
        best_p, best_m = cand, m
for ag in sorted({"max", "top2", cfg["agg"]} - {best_p["agg"]}):
    cand, m = coord_search({**best_p, "agg": ag}, free4, f"4: agg={ag}")
    if m > best_m:
        best_p, best_m = cand, m
p4 = best_p
r4 = eval_blend(p4); add_row("4. + SIFT+RANSAC Verification", r4)

abl = pd.DataFrame(ablation).set_index("stage")
print("\n============== ABLATION (pooled 5-fold OOF MAP@5) ==============")
print(abl.round(4).to_string())
print("\nfinal params:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in p4.items()})
TAUS = {sp: r4[sp]["tau"] for sp in SPECIES}
for sp in SPECIES:
    print(f"{sp}: tau={TAUS[sp]:.4f} | known-recall@5={r4[sp]['recall5']:.3f} | predicted new@1={r4[sp]['new1']:.1%} | MAP(new always last)={r4[sp]['map_new_last']:.4f}")
json.dump(dict(params=p4, w_cos=1 - p4["w_rerank"], tau=TAUS, cv_map=r4["map"], ablation=abl.round(5).reset_index().to_dict("records"),
               rerank=dict(k1=RERANK_K1, k2=RERANK_K2, lam=RERANK_LAM), sift=dict(top_m=M_SIFT, ratio=SIFT_RATIO, ransac_thresh=RANSAC_THRESH, iters=RANSAC_ITERS),
               cand_k=CAND_K, sub1_cfg=cfg),
          open("best_config_v2.json", "w"), indent=2)
print(f"saved best_config_v2.json  [{time.time()-t0:.0f}s]")

Sub #1 baseline reproduced: 0.8366  (best_config.json says 0.8366)
  [2: rerank + exact vp] 42 evals -> MAP 0.8407  w_rerank=0.203  b_same=-0.006  agg=max floor=0
  [3: + vp buckets] 51 evals -> MAP 0.8408  w_rerank=0.188  b_same=-0.006  b_adj=-0.006  b_opp=0.000  agg=max floor=0
  [4: floor=0] 100 evals -> MAP 0.8662  w_rerank=0.188  w_sift=0.147  b_same=0.006  b_adj=-0.006  b_opp=0.003  agg=max floor=0
  [4: floor=3] 101 evals -> MAP 0.8723  w_rerank=0.219  w_sift=0.144  b_same=0.044  b_adj=0.056  b_opp=0.006  agg=max floor=3
  [4: floor=5] 101 evals -> MAP 0.8700  w_rerank=0.219  w_sift=0.250  b_same=0.000  b_adj=-0.006  b_opp=0.006  agg=max floor=5
  [4: floor=8] 80 evals -> MAP 0.8641  w_rerank=0.188  w_sift=0.425  b_same=-0.006  b_adj=-0.006  b_opp=0.000  agg=max floor=8
  [4: agg=top2] 131 evals -> MAP 0.8738  w_rerank=0.281  w_sift=0.191  b_same=-0.006  b_adj=0.009  b_opp=0.013  agg=top2 floor=3

============== ABLATION (pooled 5-fold OOF MAP@5) ==============
                 

## 6. Final fit on full train → sanity checks → `submission_v2.csv`

In [11]:
cfg2, TAUS2 = json.load(open("best_config_v2.json")), TAUS
p_fin = cfg2["params"]
preds = pd.Series(index=te.index, dtype=object)
for sp in SPECIES:                                           # species never mixed
    P = POOL_TE[sp]
    top, sc = agg_topk(blend(P, p_fin), P["gc"], p_fin["agg"])
    names = np.where(top >= 0, CLASSES[sp][np.maximum(top, 0)], "")
    lists = [[n for n in l if n] for l in apply_new(names, sc, TAUS2[sp])]
    preds[te.index[P["q_rows"]]] = [" ".join(l) for l in lists]
sub = pd.DataFrame({"id": te.id.values, "predictions": preds.values})

# ---- sanity checks ----
ss = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
assert len(sub) == len(ss) == len(te), "row count"
if len(sub) != 2791: print("WARNING: expected 2791 rows, got", len(sub))
assert set(sub.id) == set(ss.id) and sub.id.is_unique, "ids differ from sample_submission"
assert not sub.predictions.isna().any(), "NaN predictions"
ident_species = tr.drop_duplicates("identity").set_index("identity").species.to_dict()
test_species = te.set_index("id").species.to_dict()
new_rank = Counter()
for i, p in zip(sub.id, sub.predictions):
    labs = p.split()
    assert 1 <= len(labs) <= 5 and len(set(labs)) == len(labs), f"bad label count/dupes in {i}"
    for l in labs:
        assert l == NEW or (l in ident_species and ident_species[l] == test_species[i]), f"invalid label {l} for {i}"
    new_rank[labs.index(NEW) + 1 if NEW in labs else 0] += 1
print("ALL SANITY CHECKS PASSED")
print("new_individual rank distribution (0 = absent):", dict(sorted(new_rank.items())))
print(f"predicted new@1 rate: {new_rank[1] / len(sub):.1%}")
print(sub.sample(10, random_state=0).to_string(index=False))
sub.to_csv(OUT_PATH, index=False); print("saved", OUT_PATH)

ALL SANITY CHECKS PASSED
new_individual rank distribution (0 = absent): {1: 1110, 2: 1433, 3: 163, 4: 48, 5: 37}
predicted new@1 rate: 39.8%
      id                                    predictions
te_02465 HYE0236 new_individual HYE0619 HYE0179 HYE0346
te_00689 new_individual LEO0391 LEO0508 LEO0549 LEO0518
te_02684 new_individual HYE0675 HYE0512 HYE0447 HYE0381
te_00992 HYE0524 new_individual HYE0034 HYE0164 HYE0577
te_00617 HYE0188 new_individual HYE0222 HYE0125 HYE0677
te_00228 new_individual LEO0147 LEO0386 LEO0546 LEO0458
te_02614 new_individual LEO0256 LEO0356 LEO0333 LEO0229
te_01666 new_individual HYE0636 HYE0662 HYE0123 HYE0468
te_00321 HYE0425 new_individual HYE0111 HYE0535 HYE0180
te_01513 HYE0177 new_individual HYE0269 HYE0603 HYE0376
saved submission.csv


### Diagnostics: TEST vs CV top-1 blended score
Compares the top-1 blended identity score of test queries with CV `val_known` / `val_new`, estimates the test share of new individuals by mixture fitting
(same method as Sub #1), and shows the test-time rate of `new_individual` at rank 1.

In [12]:
def mix_fit(T1, N, t1):
    ts = np.quantile(T1, np.linspace(.02, .98, 49))
    f_t, f_n, f_k = [(x[:, None] >= ts).mean(0) for x in (t1, T1[N], T1[~N])]
    w = float(np.clip(((f_t - f_k) * (f_n - f_k)).sum() / ((f_n - f_k) ** 2).sum(), 0, 1))
    return w, float(np.sqrt(((f_t - (w * f_n + (1 - w) * f_k)) ** 2).mean()))

det = eval_blend(p_fin, detail=True)
qs = [.05, .1, .25, .5, .75, .9]
w_est, n_te = {}, {}
for sp in SPECIES:
    d, P = det[sp], POOL_TE[sp]
    _, tsc = agg_topk(blend(P, p_fin), P["gc"], p_fin["agg"])
    T1, N = d["tsc"][:, 0], d["N"]
    w, err = mix_fit(T1, N, tsc[:, 0]); w_est[sp], n_te[sp] = w, len(tsc)
    print(f"\n=== {sp}: top-1 blended score quantiles {qs} ===")
    print("CV known :", np.round(np.quantile(T1[~N], qs), 3))
    print("CV new   :", np.round(np.quantile(T1[N], qs), 3))
    print("TEST     :", np.round(np.quantile(tsc[:, 0], qs), 3))
    print(f"estimated TEST new-individual share ~ {w:.1%} (mixture error {err:.4f}; CV assumed {N.mean():.1%})")
    print(f"test new@1 at tau={TAUS2[sp]:.3f}: {(new_position(tsc, TAUS2[sp]) == 0).mean():.1%} | CV: {(new_position(d['tsc'], TAUS2[sp]) == 0).mean():.1%}")
tot = sum(n_te.values())
print(f"\nestimated overall TEST new_individual share ~ {sum(w_est[s] * n_te[s] for s in SPECIES) / tot:.1%}")


=== hyena: top-1 blended score quantiles [0.05, 0.1, 0.25, 0.5, 0.75, 0.9] ===
CV known : [0.144 0.17  0.262 0.449 0.7   1.009]
CV new   : [0.092 0.099 0.114 0.142 0.181 0.254]
TEST     : [0.112 0.125 0.16  0.306 0.599 0.927]
estimated TEST new-individual share ~ 28.4% (mixture error 0.0112; CV assumed 32.7%)
test new@1 at tau=0.210: 37.1% | CV: 39.0%

=== leopard: top-1 blended score quantiles [0.05, 0.1, 0.25, 0.5, 0.75, 0.9] ===
CV known : [0.183 0.226 0.422 0.735 1.071 1.281]
CV new   : [0.118 0.128 0.148 0.182 0.236 0.34 ]
TEST     : [0.161 0.179 0.226 0.493 0.99  1.286]
estimated TEST new-individual share ~ 25.2% (mixture error 0.0341; CV assumed 37.6%)
test new@1 at tau=0.344: 41.9% | CV: 46.0%

estimated overall TEST new_individual share ~ 26.6%
